In [1]:
import pandas as pd
import sklearn as sk
import numpy as np
import scipy as sp
from sklearn import tree
from sklearn.metrics import confusion_matrix
from sklearn.metrics import f1_score
import time
from sklearn.ensemble import GradientBoostingClassifier
import xgboost
from xgboost.sklearn import XGBClassifier
from sklearn import preprocessing

## Data Importing 

In [3]:
# Ecom Products Classification: rightly categorizing the items based on their detailed feature specifications.
# More than 100 specifications have been collected
ecom_prod_train=pd.read_csv("https://raw.githubusercontent.com/venkatareddykonasani/Datasets/master/Ecom_Products_Menu/train.csv")
ecom_prod_test=pd.read_csv("https://raw.githubusercontent.com/venkatareddykonasani/Datasets/master/Ecom_Products_Menu/test.csv")

In [4]:
print(ecom_prod_train.sample(10))
print(ecom_prod_test.sample(10))

          id  spec1  spec2  spec3  spec4  spec5  spec6  spec7  spec8  spec9  \
32479  40122      0      0      5      0      0      0      0      0      0   
42998  53026      0      0      0      0      0      0      0      1      0   
23279  28740      2      0      0      0      0      0      0      1      0   
48527  59873      0      0      0      0      0      0      0      5      0   
34659  42787      1      1      0      0      0      0      0      0      0   
14121  17429      0      0      0      0      0      0      0      0      0   
3238    4002      0      0      0      0      0      0      0      0      0   
4722    5828      0      0      0      0      0      0      0      0      0   
33989  41956      0      0      0      0      0      0      0      0      1   
29395  36286      0      0      4      0      0      0      0      0      0   

       ...  spec92  spec93  spec94  spec95  spec96  spec97  spec98  spec99  \
32479  ...       7       0       0      28       0  

In [ ]:
# Frequency counts of the products
# This ensures that this is a multi-class classification problem
ecom_prod_train['Category'].value_counts()

Category
Tablets          13079
Personal_Care    11452
Appliances        6863
Laptops           6466
Camara            3964
Accessories       2341
Ipod              2217
TV                2182
Mobiles           1558
Name: count, dtype: int64

## Prepare Features, Train and Test data

In [6]:
features=list(ecom_prod_train.columns[1:101])
print("features \n", features)

# Defining X and y for train dataset
X_train=ecom_prod_train[features]
y_train=ecom_prod_train['Category']

# Defining X and y for test dataset
X_test=ecom_prod_test[features]
y_test=ecom_prod_test['Category']

features 
 ['spec1', 'spec2', 'spec3', 'spec4', 'spec5', 'spec6', 'spec7', 'spec8', 'spec9', 'spec10', 'spec11', 'spec12', 'spec13', 'spec14', 'spec15', 'spec16', 'spec17', 'spec18', 'spec19', 'spec20', 'spec21', 'spec22', 'spec23', 'spec24', 'spec25', 'spec26', 'spec27', 'spec28', 'spec29', 'spec30', 'spec31', 'spec32', 'spec33', 'spec34', 'spec35', 'spec36', 'spec37', 'spec38', 'spec39', 'spec40', 'spec41', 'spec42', 'spec43', 'spec44', 'spec45', 'spec46', 'spec47', 'spec48', 'spec49', 'spec50', 'spec51', 'spec52', 'spec53', 'spec54', 'spec55', 'spec56', 'spec57', 'spec58', 'spec59', 'spec60', 'spec61', 'spec62', 'spec63', 'spec64', 'spec65', 'spec66', 'spec67', 'spec68', 'spec69', 'spec70', 'spec71', 'spec72', 'spec73', 'spec74', 'spec75', 'spec76', 'spec77', 'spec78', 'spec79', 'spec80', 'spec81', 'spec82', 'spec83', 'spec84', 'spec85', 'spec86', 'spec87', 'spec88', 'spec89', 'spec90', 'spec91', 'spec92', 'spec93', 'spec94', 'spec95', 'spec96', 'spec97', 'spec98', 'spec99', 'spec10

## Decision Tree Model

In [34]:
# Buildng Decision tree on the training data to get the basic benchmark of Accuracy
from sklearn import tree
tree = tree.DecisionTreeClassifier(max_depth=9)
tree.fit(X_train,y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",9
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current 

### Decision Tree Results

In [35]:
#Accuracy on train data
print("Decision Tree Results \n")
print("Accuracy on train data" , tree.score(X_train, y_train))
print("Accuracy on test data" , tree.score(X_test, y_test))


# # Another way to compute accuracy is via confusion matrix
# from matplotlib import cm
# from sklearn.metrics import confusion_matrix

# # for train data
# y_pred_train = tree.predict(X_train)
# cm_train = confusion_matrix(y_train, y_pred_train)
# print("Confusion Matrix on train data \n", cm_train)
# ## Accuracy for train data
# total = sum(sum(cm_train))
# accuracy = (cm_train[0,0] + cm_train[1,1] + cm_train[2,2] + cm_train[3,3] + cm_train[4,4] + cm_train[5,5] + cm_train[6,6] + cm_train[7,7] + cm_train[8,8]) / total   # becuase of multiclass(9) classification
# print("Accuracy of train data : ", accuracy)

# # for test data 
# y_pred = tree.predict(X_test)
# cm_test = confusion_matrix(y_test, y_pred)
# print("Confusion Matrix for test data\n", cm_test)
# ## Accuracy for test data
# total = sum(sum(cm_test))
# accuracy = (cm_test[0,0] + cm_test[1,1] + cm_test[2,2] + cm_test[3,3] + cm_test[4,4] + cm_test[5,5] + cm_test[6,6] + cm_test[7,7] + cm_test[8,8]) / total   # becuase of multiclass(9) classification
# print("Accuracy of test data : ", accuracy)

Decision Tree Results 

Accuracy on train data 0.6363672638761422
Accuracy on test data 0.6269139162980606


## Gradient Booting Model (GBM)

In [37]:
from sklearn.ensemble import GradientBoostingClassifier
boost=GradientBoostingClassifier(n_estimators=100,learning_rate=0.1, verbose=1)   # n_estimators: number of iterations or trees
# learing_rate : shrinks the contribution of each tree by learning_rate (rho): best value is between 0.01 and 0.1
# verbose=1 : results will be shown after every iteration and verbose=0 means no output is shown
# In output, Train Loss is the remaining error after that iteration

##fitting the gradient boost classifier
start_time = time.time()
boost.fit(X_train,y_train)
print("Time taken by GBM "+ str((time.time() - start_time))+ " Seconds")

      Iter       Train Loss   Remaining Time 
         1           1.6679            1.94m
         2           1.5254            1.91m
         3           1.4205            2.72m
         4           1.3357            3.30m
         5           1.2661            3.70m
         6           1.2069            4.17m
         7           1.1570            4.35m
         8           1.1147            4.53m
         9           1.0746            4.38m
        10           1.0392            4.15m
        20           0.8356            3.75m
        30           0.7398            2.72m
        40           0.6852            2.50m
        50           0.6487            2.02m
        60           0.6220            1.48m
        70           0.6009            1.08m
        80           0.5838           41.64s
        90           0.5694           19.79s
       100           0.5563            0.00s
Time taken by GBM 189.32084369659424 Seconds


### GBM Results

In [ ]:
# Predicting Gradient boosting model on the train Data
boost_predict_train=boost.predict(X_train)
cm1 = confusion_matrix(y_train,boost_predict_train)
print(cm1)

accuracy_train=f1_score(y_train, boost_predict_train, average='micro') 
# f1_score measures how well the model distinguishes between classes
# In a single-label multi-class classification problem, the micro-averaged F1 score is identical to accuracy
# So, here f1_score with micro average gives accuracy

print("train accuracy", accuracy_train)

# Accuracy has improved from 63% to 80% for trian data after using GBM

[[ 1497   116    16    11   132    54   162    23   330]
 [   50  6328    91     5    24    74   174     0   117]
 [   21   196  3352     4     6    86   146     1   152]
 [    0     3     3  2144     7     0     4     0    56]
 [  124    18    11     1  2954     7     8    69  3274]
 [   32   225   271     9     5   767   149     2    98]
 [  108   188   126     1    13    50 10813    11   142]
 [   20     7     2    14   264     0    61   934   880]
 [   77    24    12    18  1377     8    32    94 11437]]
train accuracy 0.8025617493316308


In [ ]:
# Predicting Gradient boosting model on the test Data
boost_predict_test=boost.predict(X_test)
cm1 = confusion_matrix(y_test,boost_predict_test)
print(cm1)

accuracy_test=f1_score(y_test, boost_predict_test, average='micro') 
print("test accuracy", accuracy_test)

# # Accuracy has improved from 62% to 78% for test data after using GBM

[[ 310   31    0    1   25    9   41    5   76]
 [  14 1456   31    1    3   20   49    0   27]
 [   3   41  849    2    0   27   38    0   31]
 [   0    0    0  498    1    0    0    1   22]
 [  21    6    4    0  682    0    2   23  800]
 [   6   68   78    0    4  150   36    0   29]
 [  38   38   37    1    2   10 2512    3   42]
 [   5    1    0    3   68    0   22  169  241]
 [  20    7    7    9  329    0    8   23 2640]]
test accuracy 0.7881932630146308


## Extreme Gradient Boosting (XGB)

In [ ]:
# Creating XGB Friendly data and matrices
train_labels = y_train.values
train_labels = preprocessing.LabelEncoder().fit_transform(train_labels)
# to convert categorical text labels in train_labels into numerical integers (0 to n-1)
# fit_transform() maps each unique class to an integer in alphabetical order.
test_labels = y_test.values
test_labels = preprocessing.LabelEncoder().fit_transform(test_labels)

# we cannot directly give pandas dataframe to xgboost, so converting to DMatrix
matrix_train = xgboost.DMatrix(X_train,label=train_labels)   
matrix_test = xgboost.DMatrix(X_test,label=test_labels)

In [57]:
params = {
    'max_depth': 4,  # depth should be short in order to prolong the algo(must be atleast half of the value as that in a single Decision Tree)
    'eta':0.1, #Learning Rate
    'eval_metric':'merror', # Multiclass classification error rate (calculates the percentage of incorrect predictions made by the model)
    'tree_method' : "hist", # enables the fast histogram-optimized approximate greedy algorithm for finding the best split points when growing trees
    'num_class': 9   # number of classes in the target variable
}

start_time = time.time()

model=xgboost.train(params=params,
                    dtrain=matrix_train,
                    num_boost_round=300,    #Number of trees/iterations
                    early_stopping_rounds=4, # Stop after 4 rounds, if test error doesn't improve. 
                    evals=[(matrix_test,'test')] 
                    )

print("Time taken by XGB "+ str((time.time() - start_time))+ " Seconds")

print("Best iteration:", model.best_iteration)
print("Best merror:", model.best_score)

# output is something like [0]	test-merror:0.34986, [1]	test-merror:0.31711
# It means merror = 0.34986 → 34.986% predictions are wrong => Accuracy = 1 - merror = 65.01%
# for the next iteration/tree it's 0.31711 -> 31.711% predictions are wrong =>  Accuracy = 68.29%
# Error is decreasing → accuracy is increasing -> model is learning well

[0]	test-merror:0.34986
[1]	test-merror:0.31711
[2]	test-merror:0.30002
[3]	test-merror:0.29287
[4]	test-merror:0.28607
[5]	test-merror:0.28020
[6]	test-merror:0.27603
[7]	test-merror:0.27373
[8]	test-merror:0.27229
[9]	test-merror:0.27007
[10]	test-merror:0.26769
[11]	test-merror:0.26489
[12]	test-merror:0.26353
[13]	test-merror:0.26225
[14]	test-merror:0.26080
[15]	test-merror:0.25868
[16]	test-merror:0.25749
[17]	test-merror:0.25646
[18]	test-merror:0.25459
[19]	test-merror:0.25400
[20]	test-merror:0.25221
[21]	test-merror:0.25213
[22]	test-merror:0.25119
[23]	test-merror:0.25094
[24]	test-merror:0.25009
[25]	test-merror:0.24906
[26]	test-merror:0.24796
[27]	test-merror:0.24685
[28]	test-merror:0.24677
[29]	test-merror:0.24541
[30]	test-merror:0.24524
[31]	test-merror:0.24422
[32]	test-merror:0.24371
[33]	test-merror:0.24345
[34]	test-merror:0.24251
[35]	test-merror:0.24192
[36]	test-merror:0.24056
[37]	test-merror:0.24039
[38]	test-merror:0.23962
[39]	test-merror:0.23911
[40]	test-

### XGB Results

In [58]:
# Prediction using XGB on the train Data
boost_predict_train=model.predict(matrix_train)
cm1 = confusion_matrix(train_labels,boost_predict_train)
print(cm1)

accuracy_train=f1_score(train_labels, boost_predict_train, average='micro') 
print("train accuracy", accuracy_train)

[[ 1540   118    11    13   128    34   153    23   321]
 [   48  6354    98     4    18    66   165     0   110]
 [   19   190  3397     3     5    84   143     2   121]
 [    0     4     1  2149     7     2     3     0    51]
 [  113    18    13     0  2980     3     7    49  3283]
 [   32   239   307     5     7   746   138     2    82]
 [  124   182   143     2    18    44 10797    10   132]
 [   33     5     2    19   242     0    61   926   894]
 [   70    24    11    16  1223     6    30    77 11622]]
train accuracy 0.8082478751845497


In [59]:
# Prediction using XGB on the test Data
boost_predict_test=model.predict(matrix_test)
cm1 = confusion_matrix(test_labels,boost_predict_test)
print(cm1)

accuracy_test=f1_score(test_labels, boost_predict_test, average='micro') 
print("test accuracy", accuracy_test)

## both training and testing accuracies are quite similar to that of GBM model because both follows boosting approach
# difference is with the speed and performance optimizations

[[ 315   28    1    1   27    9   41    7   69]
 [  14 1472   29    1    0   13   43    0   29]
 [   2   42  857    2    3   19   40    0   26]
 [   0    1    0  499    0    0    0    1   21]
 [  28    5    5    0  639    0    1   14  846]
 [   8   78   83    0    5  141   33    0   23]
 [  41   37   42    1    6   10 2504    3   39]
 [   8    1    0    4   61    0   21  173  241]
 [  21    8    4    8  305    0    6   21 2670]]
test accuracy 0.7885335148009527
